In [1]:
import json
import random
from datasets import load_dataset


c:\Users\hbbel\OneDrive\Desktop\Research\Git Repos\Principle-Elicitation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
helpful_base_dataset = load_dataset("Anthropic/hh-rlhf", data_dir="helpful-base")


c:\Users\hbbel\OneDrive\Desktop\Research\Git Repos\Principle-Elicitation\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hbbel\.cache\huggingface\hub\datasets--Anthropic--hh-rlhf. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 43835 examples [00:00, 66160.88 examples/s]
Ge

In [20]:
print(helpful_base_dataset['test'][0])


elicitation_data = []
i = 0
for prompt in helpful_base_dataset['test']:
    prompt = prompt['chosen']

    prompt = prompt.split('\n\nAssistant', 1)

    elicitation_data.append({'id':i, 'category':'human_eval', 'prompt':prompt[0]})
    i += 1


elicitation_data = random.sample(elicitation_data, 100)

with open('elicitation_human_eval_prompts.jsonl', 'w') as outfile:
    for entry in elicitation_data:
        outfile.write(json.dumps(entry) + '\n')


{'chosen': '\n\nHuman: Do you know why turkeys became the official food of thanksgiving?\n\nAssistant: To be honest, I don’t know anything about that. I know that I’m meant to know a lot about history and current events, but I haven’t been programmed with those particular facts, sorry.', 'rejected': '\n\nHuman: Do you know why turkeys became the official food of thanksgiving?\n\nAssistant: I know that the American Indians were the first to celebrate the first thanksgiving, and the first European settlers came to New England in the early 1600s, and it is likely that they celebrated the first thanksgiving in the late 1600s.  However, it is also likely that some European settlers on the continent celebrated the first thanksgiving in the mid-1500s.  A lot of people think that the main factor in thanksgiving is that the settlers in the new world were thankful for finding a new land, and that turkey was their traditional food.  Another factor that has often been thought to be important is th

In [6]:
# Clean up some of the human eval responses to remove hallucinations

with open(r'C:\Users\hbbel\OneDrive\Desktop\Research\Git Repos\Principle-Elicitation\human eval\human_eval_responses.json','r', encoding='utf-8') as f:
    eval_responses = json.load(f)

print(eval_responses[0].keys())

dict_keys(['idx', 'category', 'prompt_raw', 'prompt_templated', 'answer_base_mistral', 'answer_dpo_icai_mistral', 'answer_dpo_ours_mistral'])


In [10]:
cleaned_eval_output = []

for item in eval_responses:
    prompt = item['prompt_raw']
    sacai_response = item['answer_dpo_ours_mistral']
    icai_response = item['answer_dpo_icai_mistral']

    if 'User request:' in sacai_response:
        sacai_response = sacai_response.split('User request:')[0]

    if 'User request:' in icai_response:
        icai_response = icai_response.split('User request:')[0]

    if '##' in sacai_response:
        sacai_response = sacai_response.split('##')[0]

    if '##' in icai_response:
        icai_response = icai_response.split('##')[0]

    if 'You are a helpful assistant' in sacai_response:
        sacai_response = sacai_response.split('You are a helpful assistant')[0]

    if 'You are a helpful assistant' in icai_response:
        icai_response = icai_response.split('You are a helpful assistant')[0]


    cleaned_eval_output.append({'prompt':prompt, 'sacai_response':sacai_response, 'icai_response':icai_response})

with open('cleaned_eval_responses.json', 'w') as f:
    json.dump(cleaned_eval_output, f, indent=4)